In [1]:
%pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.5 MB/s  0:00:00


In [2]:
import pandas as pd
import numpy as np
import sqlite3
from faker import Faker

rng = np.random.default_rng(42)
fake = Faker("en_GB")
Faker.seed(42)

catalogue = [
    ("PS4F",   "PlayStation 4 Fat",           500, 2013, 249.99),
    ("PS4S",   "PlayStation 4 Slim",          500, 2016, 199.99),
    ("PS4P",   "PlayStation 4 Pro",          1000, 2016, 299.99),
    ("PS5FD",  "PlayStation 5 Fat Disc",     1000, 2020, 479.99),
    ("PS5FDE", "PlayStation 5 Fat Digital",   1000, 2020, 399.99),
    ("PS5SD",  "PlayStation 5 Slim Disc",    1000, 2023, 479.99),
    ("PS5SDE", "PlayStation 5 Slim Digital", 1000, 2023, 399.99),
    ("PS5PD",  "PlayStation 5 Pro Digital",  2000, 2024, 699.99),
    ("XSS512", "Xbox Series S 512GB",         512, 2020, 249.99),
    ("XSS1T",  "Xbox Series S 1TB",          1000, 2023, 299.99),
    ("XSXD",   "Xbox Series X Disc",         1000, 2020, 499.99),
    ("XSXDE",  "Xbox Series X Digital",      1000, 2024, 449.99),
    ("NSW",    "Nintendo Switch",              32, 2017, 259.99),
    ("NSL",    "Nintendo Switch Lite",         32, 2019, 199.99),
    ("NSOLED", "Nintendo Switch OLED",          64, 2021, 309.99),
    ("NSW2",   "Nintendo Switch 2",           256, 2025, 429.99),
]

df_consoles = pd.DataFrame(
    catalogue,
    columns=[
        "Console_ID",
        "Console_Name",
        "Storage_GB",
        "Release_Year",
        "Reference_Price_GBP"
    ]
)

promotions = [
    ("NYS",  "New Year Sale 2025", "2025-01-01", "2025-01-14"),
    ("ES",   "Easter Sale 2025",    "2025-04-11", "2025-04-21"),
    ("SS",   "Summer Sale 2025",   "2025-07-01", "2025-08-31"),
    ("BF",   "Black Friday 2025",   "2025-11-20", "2025-11-30"),
    ("CS",   "Christmas Sale 2025", "2025-12-01", "2025-12-25"),
    ("NYS2", "New Year Sale 2026",   "2026-01-01", "2026-01-14"),
    ("ES2",  "Easter Sale 2026",    "2026-04-11", "2026-04-21"),
    ("SS2",  "Summer Sale 2026",    "2026-07-01", "2026-08-31"),
]

df_promotions = pd.DataFrame(
    promotions,
    columns=[
        "Promotion_ID",
        "Promotion_Name",
        "Start_Date",
        "End_Date"
    ]
)

n_customers = 1500
n_repeated_names = 15

unique_names = []

while len(unique_names) < n_customers - n_repeated_names:
    name = fake.name()

    if name not in unique_names:
        unique_names.append(name)

repeated_names = rng.choice(
    unique_names,
    size=n_repeated_names,
    replace=False
).tolist()

names = unique_names + repeated_names
rng.shuffle(names)

customers = []

for i, name in enumerate(names, start=1):
    email_name = ".".join(
        "".join(char for char in part.lower() if char.isalnum())
        for part in name.split()
    )

    email = f"{email_name}.{i:04d}@example.com"

    age = int(rng.integers(15, 51))

    if age <= 17:
        age_group = "Teenager"
    elif age <= 24:
        age_group = "Young Adult"
    elif age <= 34:
        age_group = "Adult"
    elif age <= 44:
        age_group = "Middle-aged Adult"
    else:
        age_group = "Older Adult"


    customers.append([
        f"U{i:04d}",
        name,
        email,
        age,
        age_group,
        fake.postcode()
    ])

df_customers = pd.DataFrame(
    customers,
    columns=[
        "Customer_ID",
        "Customer_Name",
        "Email",
        "Age",
        "Age_Group",
        "Postcode"
    ]
)

weights = {
    2025: np.array([
        2, 2, 2, 10, 5, 12, 7, 6,
        7, 5, 8, 4, 5, 3, 5, 7
    ], dtype=float),

    2026: np.array([
        0.5, 0.5, 0.5, 10, 5, 12, 7, 8,
        7, 6, 8, 5, 4, 2, 4, 12
    ], dtype=float)
}

sales = []
serial_counts = {}

for year in [2025, 2026]:
    start_date = pd.Timestamp(f"{year}-01-01")
    end_date = pd.Timestamp(
        "2025-12-31" if year == 2025 else "2026-10-25"
    )

    for _ in range(750):
        # Generate a random sale date
        sale_date = start_date + pd.Timedelta(
            days=int(rng.integers(0, (end_date - start_date).days + 1))
        )
        date_text = sale_date.strftime("%Y-%m-%d")

        # Select consoles released by the sale date
        eligible = df_consoles[
            df_consoles["Release_Year"] <= year
        ]

        if sale_date < pd.Timestamp("2025-06-05"):
            eligible = eligible[eligible["Console_ID"] != "NSW2"]

        eligible_indices = eligible.index.to_numpy()
        probabilities = weights[year][eligible_indices]
        probabilities /= probabilities.sum()

        console_index = rng.choice(
            eligible_indices, p=probabilities
        )
        console = df_consoles.loc[console_index]
        console_id = console["Console_ID"]

        # Create unique sale and device serial numbers
        sale_number = len(sales) + 1
        serial_counts[console_id] = serial_counts.get(console_id, 0) + 1
        serial_number = 10000 + serial_counts[console_id]

        # Find any promotion active on the sale date
        active = df_promotions[
            (df_promotions["Start_Date"] <= date_text) &
            (df_promotions["End_Date"] >= date_text)
        ]

        promotion_id = None
        discount = 0

        if not active.empty:
            promotion_id = rng.choice(active["Promotion_ID"].to_numpy())
            discount = int(rng.choice([5, 10, 15, 20]))

        # Calculate final selling price
        unit_price = float(console["Reference_Price_GBP"])

        if year == 2026:
            unit_price *= 1.15

        unit_price = round(unit_price * (1 - discount / 100), 2)

        sales.append([
            f"S{sale_number:05d}",
            console_id,
            f"U{sale_number:04d}",
            promotion_id,
            date_text,
            unit_price,
            discount,
            serial_number
        ])

df_sales = pd.DataFrame(
    sales,
    columns=[
        "Sale_ID",
        "Console_ID",
        "Customer_ID",
        "Promotion_ID",
        "Sale_Date",
        "Unit_Price_GBP",
        "Discount_Percent",
        "Serial_Number"
    ]
)

# Deliberately introduce missing Promotion_ID values in 2% of sales
missing_rows = rng.choice(
    df_sales.index,
    size=int(len(df_sales) * 0.02),
    replace=False
)

df_sales.loc[missing_rows, "Promotion_ID"] = None

df_consoles.to_csv("Consoles.csv", index=False)
df_promotions.to_csv("Promotions.csv", index=False)
df_customers.to_csv("Customers.csv", index=False)
df_sales.to_csv("Sales.csv", index=False)

print("CSV files saved.")


# create table for sqlite

conn = sqlite3.connect("gaming_sales.db")
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
DROP TABLE IF EXISTS Sales;
DROP TABLE IF EXISTS Customers;
DROP TABLE IF EXISTS Promotions;
DROP TABLE IF EXISTS Consoles;

CREATE TABLE Consoles (
    Console_ID TEXT PRIMARY KEY,
    Console_Name TEXT,
    Storage_GB INTEGER,
    Release_Year INTEGER,
    Reference_Price_GBP REAL
);

CREATE TABLE Promotions (
    Promotion_ID TEXT PRIMARY KEY,
    Promotion_Name TEXT,
    Start_Date TEXT,
    End_Date TEXT
);

CREATE TABLE Customers (
    Customer_ID TEXT PRIMARY KEY,
    Customer_Name TEXT,
    Email TEXT UNIQUE,
    Age INTEGER,
    Age_Group TEXT,
    Postcode TEXT
);

CREATE TABLE Sales (
    Sale_ID TEXT PRIMARY KEY,
    Console_ID TEXT,
    Customer_ID TEXT,
    Promotion_ID TEXT,
    Sale_Date TEXT,
    Unit_Price_GBP REAL,
    Discount_Percent INTEGER,
    Serial_Number INTEGER,

    FOREIGN KEY (Console_ID)
        REFERENCES Consoles(Console_ID),

    FOREIGN KEY (Customer_ID)
        REFERENCES Customers(Customer_ID),

    FOREIGN KEY (Promotion_ID)
        REFERENCES Promotions(Promotion_ID),

    UNIQUE (Console_ID, Serial_Number)
);
""")


# import data into sqlite
df_consoles.to_sql(
    "Consoles", conn, if_exists="append", index=False
)

df_promotions.to_sql(
    "Promotions", conn, if_exists="append", index=False
)

df_customers.to_sql(
    "Customers", conn, if_exists="append", index=False
)

df_sales.to_sql(
    "Sales", conn, if_exists="append", index=False
)

conn.commit()

#checking the connection

print(
    pd.read_sql_query(
        "SELECT * FROM Sales LIMIT 5",
        conn
    )
)

conn.close()

CSV files saved.
  Sale_ID Console_ID Customer_ID Promotion_ID   Sale_Date  Unit_Price_GBP  \
0  S00001     PS5SDE       U0001         None  2025-03-24          399.99   
1  S00002     NSOLED       U0002           ES  2025-04-21          278.99   
2  S00003     NSOLED       U0003         None  2025-12-26          309.99   
3  S00004      PS5SD       U0004           SS  2025-08-01          383.99   
4  S00005      PS5PD       U0005          NYS  2025-01-14          629.99   

   Discount_Percent  Serial_Number  
0                 0          10001  
1                10          10001  
2                 0          10002  
3                20          10001  
4                10          10001  
